# Embedding Visualization: CLIP vs Fine-tuned Models

This notebook visualizes and compares embeddings from different CLIP models.

**Workflow:**
1. First, run `compute_embeddings.py` for each model you want to compare
2. Load the saved embeddings in this notebook
3. Explore the visualizations interactively

**Example commands to prepare embeddings:**
```bash
# Baseline CLIP
python experiments/compute_embeddings.py \
    --json_folder swap_pos_json/coco_train/ \
    --image_root . \
    --output_dir embeddings/clip_baseline \
    --model_name "CLIP Baseline" \
    --num_samples 100

# Fine-tuned model
python experiments/compute_embeddings.py \
    --json_folder swap_pos_json/coco_train/ \
    --image_root . \
    --checkpoint_path /path/to/finetuned.pt \
    --checkpoint_type external \
    --output_dir embeddings/finetuned \
    --model_name "Our Model"
```

In [1]:
# Imports
import os
import sys
import json
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image

# Interactive widgets
from IPython.display import display, HTML
import ipywidgets as widgets

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 100,
})

print("Imports complete!")

Imports complete!


## 1. Configuration

Set the paths to your saved embeddings.

In [ ]:
# Configure paths to embedding directories
EMBEDDING_DIRS = {
    'CLIP Baseline': 'embeddings/clip_baseline',
    # 'Our Model': 'embeddings/finetuned',
    # Add more models as needed
}

# Color scheme
COLORS = {
    'image': '#2E86AB',
    'caption': '#343A40',
    'component_pos': '#28A745',
    'component_neg': '#DC3545',
    'relation_correct': '#6F42C1',
    'relation_swapped': '#FD7E14',
    'relation_neg': '#FF6B6B',
    'binding_pos': '#17A2B8',
    'binding_neg': '#E83E8C',
}

MARKERS = {
    'image': 's',
    'caption': 'p',
    'component_pos': 'o',
    'component_neg': 'X',
    'relation_correct': '^',
    'relation_swapped': 'v',
    'relation_neg': '<',
    'binding_pos': 'D',
    'binding_neg': 'd',
}

## 2. Load Embeddings

In [ ]:
def load_embeddings(embedding_dir: str) -> Dict:
    """Load saved embeddings and metadata."""
    embedding_dir = Path(embedding_dir)
    
    # Load embeddings
    emb_data = np.load(embedding_dir / "embeddings.npz")
    
    # Load metadata
    with open(embedding_dir / "metadata.json", 'r') as f:
        metadata = json.load(f)
    
    # Check for thumbnails
    thumbnail_dir = embedding_dir / "thumbnails"
    has_thumbnails = thumbnail_dir.exists()
    
    return {
        'image_embeddings': emb_data['image_embeddings'],
        'text_embeddings': emb_data['text_embeddings'],
        'thumbnail_dir': thumbnail_dir if has_thumbnails else None,
        **metadata,
    }

# Load all models
models_data = {}
for name, path in EMBEDDING_DIRS.items():
    if os.path.exists(path):
        print(f"Loading {name} from {path}...")
        models_data[name] = load_embeddings(path)
        print(f"  ✓ {models_data[name]['n_samples']} samples, {models_data[name]['n_texts']} texts")
    else:
        print(f"  ✗ Path not found: {path}")

print(f"\nLoaded {len(models_data)} models")

## 3. Helper Functions

In [ ]:
def reduce_dimensions(embeddings: np.ndarray, method: str = 'umap', **kwargs) -> np.ndarray:
    """Reduce embeddings to 2D."""
    if len(embeddings) < 5:
        return None
    
    if method == 'umap':
        try:
            import umap
            n_neighbors = min(kwargs.get('n_neighbors', 15), len(embeddings) - 1)
            reducer = umap.UMAP(
                n_components=2,
                n_neighbors=n_neighbors,
                min_dist=kwargs.get('min_dist', 0.1),
                random_state=42,
                metric='cosine'
            )
            return reducer.fit_transform(embeddings)
        except ImportError:
            print("UMAP not available, falling back to t-SNE")
            method = 'tsne'
    
    if method == 'tsne':
        from sklearn.manifold import TSNE
        perplexity = min(kwargs.get('perplexity', 30), len(embeddings) - 1)
        reducer = TSNE(n_components=2, perplexity=perplexity, random_state=42)
        return reducer.fit_transform(embeddings)
    
    if method == 'pca':
        from sklearn.decomposition import PCA
        reducer = PCA(n_components=2)
        return reducer.fit_transform(embeddings)
    
    return None


def get_sample_data(data: Dict, sample_idx: int) -> Dict:
    """Get all data for a specific sample."""
    text_indices = data['text_indices_per_sample'][sample_idx]
    
    return {
        'sample_id': data['sample_ids'][sample_idx],
        'image_path': data['image_paths'][sample_idx],
        'image_emb': data['image_embeddings'][sample_idx],
        'texts': [data['texts'][i] for i in text_indices],
        'text_embs': data['text_embeddings'][text_indices],
        'categories': [data['categories'][i] for i in text_indices],
        'is_positive': [data['is_positive'][i] for i in text_indices],
    }


def compute_similarities(image_emb: np.ndarray, text_embs: np.ndarray) -> np.ndarray:
    """Compute cosine similarities."""
    return np.dot(text_embs, image_emb)


def load_thumbnail(data: Dict, sample_idx: int, size: int = 100) -> Optional[Image.Image]:
    """Load thumbnail for a sample."""
    if data['thumbnail_dir']:
        thumb_path = Path(data['thumbnail_dir']) / f"sample_{sample_idx:04d}.jpg"
        if thumb_path.exists():
            return Image.open(thumb_path)
    
    # Fall back to original image
    try:
        img = Image.open(data['image_paths'][sample_idx])
        img.thumbnail((size, size))
        return img
    except:
        return None

## 4. Joint Image-Text Embedding Space

Visualize how images and texts cluster together in embedding space.

In [ ]:
def plot_joint_embedding_space(
    data: Dict,
    method: str = 'umap',
    title: str = 'Joint Image-Text Embedding Space',
    figsize: Tuple[int, int] = (14, 10),
    show_legend: bool = True,
):
    """Plot image and text embeddings together."""
    # Combine all embeddings
    all_embs = np.vstack([data['image_embeddings'], data['text_embeddings']])
    
    # Reduce to 2D
    print(f"Reducing {len(all_embs)} embeddings with {method}...")
    embs_2d = reduce_dimensions(all_embs, method=method)
    
    if embs_2d is None:
        print("Failed to reduce dimensions")
        return
    
    n_images = len(data['image_embeddings'])
    image_2d = embs_2d[:n_images]
    text_2d = embs_2d[n_images:]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot image embeddings
    ax.scatter(
        image_2d[:, 0], image_2d[:, 1],
        c=COLORS['image'], marker=MARKERS['image'],
        s=100, alpha=0.8, edgecolors='white', linewidths=0.5,
        label='Image embeddings', zorder=10,
    )
    
    # Plot text embeddings by category
    categories = list(set(data['categories']))
    for cat in categories:
        mask = [c == cat for c in data['categories']]
        points = text_2d[mask]
        
        is_negative = 'neg' in cat or 'swapped' in cat
        
        ax.scatter(
            points[:, 0], points[:, 1],
            c=COLORS.get(cat, 'gray'),
            marker=MARKERS.get(cat, 'o'),
            s=40 if is_negative else 60,
            alpha=0.5 if is_negative else 0.7,
            label=cat.replace('_', ' ').title(),
            zorder=5 if is_negative else 7,
        )
    
    ax.set_xlabel(f"{method.upper()} Dimension 1", fontsize=12)
    ax.set_ylabel(f"{method.upper()} Dimension 2", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    if show_legend:
        ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    
    plt.tight_layout()
    return fig

# Plot for first model
if models_data:
    model_name = list(models_data.keys())[0]
    fig = plot_joint_embedding_space(
        models_data[model_name],
        method='pca',  # Use PCA for speed, change to 'umap' for better visualization
        title=f'Joint Embedding Space - {model_name}'
    )
    plt.show()

## 5. Similarity Distribution by Category

Compare how similar positive vs negative texts are to images.

In [ ]:
def compute_all_similarities(data: Dict) -> Dict[str, List[float]]:
    """Compute similarities for all samples by category."""
    cat_sims = {}
    
    for sample_idx in range(data['n_samples']):
        sample = get_sample_data(data, sample_idx)
        sims = compute_similarities(sample['image_emb'], sample['text_embs'])
        
        for cat, sim in zip(sample['categories'], sims):
            if cat not in cat_sims:
                cat_sims[cat] = []
            cat_sims[cat].append(sim)
    
    return cat_sims


def plot_similarity_distribution(data: Dict, title: str = 'Similarity Distribution'):
    """Plot similarity distributions by category."""
    cat_sims = compute_all_similarities(data)
    
    # Sort: positives first, then negatives
    pos_cats = [c for c in cat_sims if 'pos' in c or 'correct' in c or c == 'caption']
    neg_cats = [c for c in cat_sims if 'neg' in c or 'swapped' in c]
    sorted_cats = pos_cats + neg_cats
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    positions = range(len(sorted_cats))
    data_to_plot = [cat_sims[c] for c in sorted_cats]
    colors = [COLORS.get(c, 'gray') for c in sorted_cats]
    
    # Violin plot
    parts = ax.violinplot(data_to_plot, positions=positions, showmeans=True, showmedians=True)
    
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.7)
    
    ax.set_xticks(positions)
    ax.set_xticklabels([c.replace('_', '\n') for c in sorted_cats], fontsize=9)
    ax.set_ylabel('Similarity to Image')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add mean annotations
    for i, data_cat in enumerate(data_to_plot):
        mean_val = np.mean(data_cat)
        ax.annotate(f'μ={mean_val:.2f}', (i, mean_val + 0.03), 
                   ha='center', fontsize=8, fontweight='bold')
    
    # Dividing line
    if pos_cats and neg_cats:
        ax.axvline(x=len(pos_cats) - 0.5, color='gray', linestyle='--', alpha=0.5)
        ax.text(len(pos_cats)/2 - 0.5, 0.98, 'POSITIVES', ha='center', 
               fontsize=10, color='green', fontweight='bold')
        ax.text(len(pos_cats) + len(neg_cats)/2 - 0.5, 0.98, 'NEGATIVES', ha='center',
               fontsize=10, color='red', fontweight='bold')
    
    plt.tight_layout()
    return fig

# Plot
if models_data:
    model_name = list(models_data.keys())[0]
    fig = plot_similarity_distribution(
        models_data[model_name],
        title=f'Similarity Distribution - {model_name}'
    )
    plt.show()

## 6. Sample Neighborhood Viewer

Interactive widget to explore individual samples with their images.

In [ ]:
def plot_sample_neighborhood(data: Dict, sample_idx: int, figsize: Tuple[int, int] = (16, 6)):
    """Plot a single sample's embedding neighborhood with the actual image."""
    sample = get_sample_data(data, sample_idx)
    sims = compute_similarities(sample['image_emb'], sample['text_embs'])
    
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 3, width_ratios=[1, 1.5, 1], wspace=0.3)
    
    # Left: Image
    ax_img = fig.add_subplot(gs[0])
    thumb = load_thumbnail(data, sample_idx, size=300)
    if thumb:
        ax_img.imshow(thumb)
    ax_img.set_title(f"Sample: {sample['sample_id'][:20]}...", fontsize=11, fontweight='bold')
    ax_img.axis('off')
    
    # Middle: Embedding space (PCA)
    ax_emb = fig.add_subplot(gs[1])
    
    all_embs = np.vstack([[sample['image_emb']], sample['text_embs']])
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    embs_2d = pca.fit_transform(all_embs)
    
    image_2d = embs_2d[0]
    text_2d = embs_2d[1:]
    
    # Plot image point
    ax_emb.scatter([image_2d[0]], [image_2d[1]], c=COLORS['image'], marker='s', s=300,
                  edgecolors='black', linewidths=2, label='Image', zorder=20)
    
    # Plot text points with connections
    for i, (cat, text, sim) in enumerate(zip(sample['categories'], sample['texts'], sims)):
        point = text_2d[i]
        is_pos = sample['is_positive'][i]
        
        ax_emb.scatter([point[0]], [point[1]], c=COLORS.get(cat, 'gray'),
                      marker=MARKERS.get(cat, 'o'), s=100, alpha=0.8, zorder=10)
        
        # Connection line
        ax_emb.plot([image_2d[0], point[0]], [image_2d[1], point[1]],
                   color=COLORS.get(cat, 'gray'), alpha=0.2 + 0.5 * sim,
                   linewidth=1 + 2 * sim, linestyle='-' if is_pos else '--', zorder=1)
        
        # Text label
        short_text = text[:20] + "..." if len(text) > 20 else text
        ax_emb.annotate(f"{short_text}\n({sim:.2f})", point, fontsize=7,
                       xytext=(5, 5), textcoords='offset points',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    ax_emb.set_title('Embedding Neighborhood', fontsize=11, fontweight='bold')
    ax_emb.grid(True, alpha=0.3)
    
    # Right: Similarity bar chart
    ax_sim = fig.add_subplot(gs[2])
    
    sorted_idx = np.argsort(sims)[::-1]
    bar_colors = [COLORS.get(sample['categories'][i], 'gray') for i in sorted_idx]
    bar_labels = [sample['texts'][i][:18] + "..." if len(sample['texts'][i]) > 18 
                  else sample['texts'][i] for i in sorted_idx]
    bar_values = [sims[i] for i in sorted_idx]
    
    y_pos = np.arange(len(bar_values))
    bars = ax_sim.barh(y_pos, bar_values, color=bar_colors, alpha=0.8)
    ax_sim.set_yticks(y_pos)
    ax_sim.set_yticklabels(bar_labels, fontsize=8)
    ax_sim.set_xlabel('Similarity')
    ax_sim.set_title('Similarity Ranking', fontsize=11, fontweight='bold')
    ax_sim.set_xlim(0, 1)
    ax_sim.invert_yaxis()
    
    for bar, val in zip(bars, bar_values):
        ax_sim.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{val:.2f}', va='center', fontsize=8)
    
    plt.tight_layout()
    return fig


# Interactive widget
if models_data:
    model_name = list(models_data.keys())[0]
    data = models_data[model_name]
    
    @widgets.interact(sample_idx=widgets.IntSlider(min=0, max=data['n_samples']-1, value=0, description='Sample:'))
    def show_sample(sample_idx):
        fig = plot_sample_neighborhood(data, sample_idx)
        plt.show()

## 7. Model Comparison

Compare embeddings from multiple models side-by-side.

In [ ]:
def compare_models_similarity_distribution(models_data: Dict[str, Dict]):
    """Compare similarity distributions across models."""
    if len(models_data) < 2:
        print("Need at least 2 models for comparison")
        return
    
    # Compute similarities for each model
    all_cat_sims = {}
    for model_name, data in models_data.items():
        all_cat_sims[model_name] = compute_all_similarities(data)
    
    # Get common categories
    categories = list(set.intersection(*[set(cs.keys()) for cs in all_cat_sims.values()]))
    categories = sorted(categories, key=lambda x: ('neg' in x or 'swapped' in x, x))
    
    # Create plot
    n_cats = len(categories)
    n_models = len(models_data)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    width = 0.8 / n_models
    x = np.arange(n_cats)
    
    model_colors = plt.cm.Set2(np.linspace(0, 1, n_models))
    
    for i, (model_name, cat_sims) in enumerate(all_cat_sims.items()):
        means = [np.mean(cat_sims[cat]) for cat in categories]
        stds = [np.std(cat_sims[cat]) for cat in categories]
        
        offset = (i - n_models/2 + 0.5) * width
        bars = ax.bar(x + offset, means, width, yerr=stds, capsize=2,
                     label=model_name, color=model_colors[i], alpha=0.8)
    
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace('_', '\n') for c in categories], fontsize=9)
    ax.set_ylabel('Mean Similarity to Image')
    ax.set_title('Model Comparison: Similarity by Category', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    return fig

# Plot comparison
if len(models_data) >= 2:
    fig = compare_models_similarity_distribution(models_data)
    plt.show()
else:
    print("Load at least 2 models to compare. Update EMBEDDING_DIRS in cell 2.")

In [ ]:
def compare_sample_across_models(models_data: Dict[str, Dict], sample_idx: int):
    """Compare a single sample across multiple models."""
    n_models = len(models_data)
    
    fig = plt.figure(figsize=(6 * n_models, 8))
    
    # Get common sample data
    first_model = list(models_data.values())[0]
    thumb = load_thumbnail(first_model, sample_idx, size=200)
    sample_id = first_model['sample_ids'][sample_idx]
    
    gs = GridSpec(2, n_models, height_ratios=[1, 2], hspace=0.3)
    
    # Top row: Image (shared)
    ax_img = fig.add_subplot(gs[0, :])
    if thumb:
        ax_img.imshow(thumb)
    ax_img.set_title(f"Sample: {sample_id[:30]}...", fontsize=12, fontweight='bold')
    ax_img.axis('off')
    
    # Bottom row: Similarity bars for each model
    for i, (model_name, data) in enumerate(models_data.items()):
        ax = fig.add_subplot(gs[1, i])
        
        sample = get_sample_data(data, sample_idx)
        sims = compute_similarities(sample['image_emb'], sample['text_embs'])
        
        sorted_idx = np.argsort(sims)[::-1]
        bar_colors = [COLORS.get(sample['categories'][j], 'gray') for j in sorted_idx]
        bar_labels = [sample['texts'][j][:15] + "..." if len(sample['texts'][j]) > 15
                      else sample['texts'][j] for j in sorted_idx]
        bar_values = [sims[j] for j in sorted_idx]
        
        y_pos = np.arange(len(bar_values))
        bars = ax.barh(y_pos, bar_values, color=bar_colors, alpha=0.8)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(bar_labels, fontsize=8)
        ax.set_xlabel('Similarity')
        ax.set_title(model_name, fontsize=11, fontweight='bold')
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        
        for bar, val in zip(bars, bar_values):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{val:.2f}', va='center', fontsize=7)
    
    plt.tight_layout()
    return fig

# Interactive comparison
if len(models_data) >= 2:
    first_model = list(models_data.values())[0]
    
    @widgets.interact(sample_idx=widgets.IntSlider(min=0, max=first_model['n_samples']-1, value=0, description='Sample:'))
    def compare_sample(sample_idx):
        fig = compare_sample_across_models(models_data, sample_idx)
        plt.show()
else:
    print("Load at least 2 models to compare.")

## 8. Accuracy Metrics

Compute and compare accuracy (positive > negative) across models.

In [ ]:
def compute_accuracy_metrics(data: Dict) -> Dict[str, float]:
    """Compute accuracy metrics for a model."""
    metrics = {
        'component_correct': 0,
        'component_total': 0,
        'relation_correct': 0,
        'relation_total': 0,
        'binding_correct': 0,
        'binding_total': 0,
    }
    
    for sample_idx in range(data['n_samples']):
        sample = get_sample_data(data, sample_idx)
        sims = compute_similarities(sample['image_emb'], sample['text_embs'])
        
        # Group by category
        cat_sims = {}
        for cat, sim in zip(sample['categories'], sims):
            if cat not in cat_sims:
                cat_sims[cat] = []
            cat_sims[cat].append(sim)
        
        # Component accuracy
        comp_pos = cat_sims.get('component_pos', [])
        comp_neg = cat_sims.get('component_neg', [])
        for pos_sim in comp_pos:
            for neg_sim in comp_neg:
                metrics['component_total'] += 1
                if pos_sim > neg_sim:
                    metrics['component_correct'] += 1
        
        # Relation accuracy
        for correct_sim, swapped_sim in zip(cat_sims.get('relation_correct', []), 
                                            cat_sims.get('relation_swapped', [])):
            metrics['relation_total'] += 1
            if correct_sim > swapped_sim:
                metrics['relation_correct'] += 1
        
        # Binding accuracy
        for pos_sim, neg_sim in zip(cat_sims.get('binding_pos', []),
                                    cat_sims.get('binding_neg', [])):
            metrics['binding_total'] += 1
            if pos_sim > neg_sim:
                metrics['binding_correct'] += 1
    
    return {
        'component_acc': metrics['component_correct'] / max(1, metrics['component_total']),
        'relation_acc': metrics['relation_correct'] / max(1, metrics['relation_total']),
        'binding_acc': metrics['binding_correct'] / max(1, metrics['binding_total']),
    }


# Compute and display metrics
print("\n" + "="*70)
print("ACCURACY METRICS (positive similarity > negative similarity)")
print("="*70)
print(f"{'Model':<25} {'Component':>15} {'Relation':>15} {'Binding':>15}")
print("-"*70)

all_metrics = {}
for model_name, data in models_data.items():
    metrics = compute_accuracy_metrics(data)
    all_metrics[model_name] = metrics
    print(f"{model_name:<25} {metrics['component_acc']:>15.1%} {metrics['relation_acc']:>15.1%} {metrics['binding_acc']:>15.1%}")

print("="*70)

In [ ]:
# Plot accuracy comparison
if all_metrics:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    metric_names = ['component_acc', 'relation_acc', 'binding_acc']
    display_names = ['Component', 'Relation', 'Binding']
    x = np.arange(len(metric_names))
    width = 0.8 / len(all_metrics)
    
    colors = plt.cm.Set2(np.linspace(0, 1, len(all_metrics)))
    
    for i, (model_name, metrics) in enumerate(all_metrics.items()):
        values = [metrics[m] for m in metric_names]
        offset = (i - len(all_metrics)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model_name, color=colors[i], alpha=0.8)
        
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                   f'{val:.1%}', ha='center', fontsize=9)
    
    ax.set_xticks(x)
    ax.set_xticklabels(display_names)
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy Comparison: Positive > Negative', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_ylim(0, 1.1)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 9. Gallery View

View multiple samples at once with their embedding visualizations.

In [ ]:
def plot_gallery(data: Dict, start_idx: int = 0, n_samples: int = 6):
    """Plot a gallery of samples with mini embedding plots."""
    n_cols = 3
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(18, 4 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(n_samples):
        sample_idx = start_idx + idx
        if sample_idx >= data['n_samples']:
            break
        
        row = idx // n_cols
        col = (idx % n_cols) * 2
        
        sample = get_sample_data(data, sample_idx)
        sims = compute_similarities(sample['image_emb'], sample['text_embs'])
        
        # Image
        ax_img = axes[row, col]
        thumb = load_thumbnail(data, sample_idx)
        if thumb:
            ax_img.imshow(thumb)
        ax_img.set_title(f"{sample['sample_id'][:15]}...", fontsize=9)
        ax_img.axis('off')
        
        # Mini embedding plot
        ax_emb = axes[row, col + 1]
        
        all_embs = np.vstack([[sample['image_emb']], sample['text_embs']])
        from sklearn.decomposition import PCA
        pca = PCA(n_components=2)
        embs_2d = pca.fit_transform(all_embs)
        
        image_2d = embs_2d[0]
        text_2d = embs_2d[1:]
        
        ax_emb.scatter([image_2d[0]], [image_2d[1]], c=COLORS['image'], marker='s', s=120,
                      edgecolors='black', linewidths=1, zorder=20)
        
        for i, cat in enumerate(sample['categories']):
            ax_emb.scatter([text_2d[i, 0]], [text_2d[i, 1]], 
                          c=COLORS.get(cat, 'gray'), marker=MARKERS.get(cat, 'o'),
                          s=40, alpha=0.7, zorder=10)
            ax_emb.plot([image_2d[0], text_2d[i, 0]], [image_2d[1], text_2d[i, 1]],
                       color=COLORS.get(cat, 'gray'), alpha=0.2, linewidth=0.8)
        
        ax_emb.grid(True, alpha=0.3)
        ax_emb.set_xticks([])
        ax_emb.set_yticks([])
    
    # Hide unused axes
    for idx in range(n_samples, n_rows * n_cols):
        row = idx // n_cols
        col = (idx % n_cols) * 2
        axes[row, col].axis('off')
        axes[row, col + 1].axis('off')
    
    plt.suptitle(f"Sample Gallery (starting at {start_idx})", fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

# Interactive gallery
if models_data:
    model_name = list(models_data.keys())[0]
    data = models_data[model_name]
    
    @widgets.interact(start_idx=widgets.IntSlider(min=0, max=max(0, data['n_samples']-6), value=0, step=6, description='Start:'))
    def show_gallery(start_idx):
        fig = plot_gallery(data, start_idx)
        plt.show()

## 10. Export Visualizations

Save publication-ready figures.

In [ ]:
# Create output directory
output_dir = Path("visualization_outputs")
output_dir.mkdir(exist_ok=True)

def save_all_figures(models_data: Dict[str, Dict], output_dir: Path):
    """Save all key figures."""
    print("Saving figures...")
    
    for model_name, data in models_data.items():
        safe_name = model_name.replace(' ', '_').replace('/', '-')
        
        # Joint embedding space
        print(f"  {model_name}: Joint embedding space...")
        fig = plot_joint_embedding_space(data, method='pca', 
                                        title=f'Joint Embedding Space - {model_name}')
        if fig:
            fig.savefig(output_dir / f"{safe_name}_joint_embedding.png", dpi=150)
            plt.close(fig)
        
        # Similarity distribution
        print(f"  {model_name}: Similarity distribution...")
        fig = plot_similarity_distribution(data, title=f'Similarity Distribution - {model_name}')
        if fig:
            fig.savefig(output_dir / f"{safe_name}_similarity_dist.png", dpi=150)
            plt.close(fig)
        
        # Sample neighborhoods (first 5)
        for i in range(min(5, data['n_samples'])):
            fig = plot_sample_neighborhood(data, i)
            if fig:
                fig.savefig(output_dir / f"{safe_name}_sample_{i:02d}.png", dpi=150)
                plt.close(fig)
    
    # Model comparison (if multiple models)
    if len(models_data) >= 2:
        print("  Model comparison...")
        fig = compare_models_similarity_distribution(models_data)
        if fig:
            fig.savefig(output_dir / "model_comparison.png", dpi=150)
            plt.close(fig)
    
    print(f"\n✅ Figures saved to {output_dir}/")

# Uncomment to save all figures
# save_all_figures(models_data, output_dir)